In [3]:
!pip install wget
!apt-get install sox libsndfile1 ffmpeg libsox-fmt-mp3 jq
!pip install text-unidecode
!pip install matplotlib>=3.3.2
!pip install Cython
!pip3 install --no-cache-dir huggingface-hub==0.23.2


'apt-get' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [4]:
import platform
print(platform.system())

Windows


In [2]:
BRANCH = "v1.23.0"

!python -m pip install "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@{BRANCH}"

  Cloning https://github.com/NVIDIA/NeMo.git (to revision v1.23.0) to C:\Users\osaoudi\AppData\Local\Temp\pip-install-s1d_t82y\nemo-toolkit_b0b225b07f1d4c94838d3b668ef4aeb2
  Resolved https://github.com/NVIDIA/NeMo.git to commit d2283e3620cd7f99dbe29fdf079757ab9f6cdf01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
INFO: pip is looking at multiple versions of nemo-toolkit to determine which version is compatible with other requirements. This could take a while.


  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/NeMo.git 'C:\Users\osaoudi\AppData\Local\Temp\pip-install-s1d_t82y\nemo-toolkit_b0b225b07f1d4c94838d3b668ef4aeb2'
  Running command git checkout -q d2283e3620cd7f99dbe29fdf079757ab9f6cdf01
ERROR: Could not find a version that satisfies the requirement triton (from nemo-toolkit) (from versions: none)
ERROR: No matching distribution found for triton


In [3]:
import urllib.request

url = "https://dldata-public.s3.us-east-2.amazonaws.com/an4_sphere.tar.gz"
urllib.request.urlretrieve(url, "an4_sphere.tar.gz")

print("Download complete. Extracting files...")

Download complete. Extracting files...


In [4]:
import os
import tarfile
import shutil

# Current working directory
DATA_DIR = os.getcwd()
os.environ["DATA_DIR"] = DATA_DIR

# Extract the tar.gz file
with tarfile.open("an4_sphere.tar.gz", "r:gz") as tar:
    tar.extractall(path=DATA_DIR)

print("Extraction complete!")

# Verify the an4 folder exists
an4_path = os.path.join(DATA_DIR, "an4")
print("AN4 path:", an4_path)

Extraction complete!
AN4 path: c:\Users\osaoudi\Desktop\EASPORTS\VoiceComAnalysis\FineTune\an4


In [5]:
import json
import librosa
import os
import glob
import subprocess

source_data_dir = f"{DATA_DIR}/an4"
target_data_dir = f"{DATA_DIR}/an4_converted"


def an4_build_manifest(transcripts_path, manifest_path, target_wavs_dir):
    """Build AN4 manifest"""
    
    with open(transcripts_path, "r") as fin:
        with open(manifest_path, "w") as fout:
            
            for line in fin:
                transcript = line[: line.find("(") - 1].lower()
                transcript = transcript.replace("<s>", "")
                transcript = transcript.replace("</s>", "")
                transcript = transcript.strip()

                file_id = line[line.find("(") + 1 : -2]

                audio_path = os.path.join(
                    target_wavs_dir,
                    file_id + ".wav"
                )

                duration = librosa.get_duration(path=audio_path)

                metadata = {
                    "audio_filepath": audio_path,
                    "duration": duration,
                    "text": transcript,
                }

                json.dump(metadata, fout)
                fout.write("\n")


# ------------------------------------------------------------------
# Check dataset
# ------------------------------------------------------------------
if not os.path.exists(source_data_dir):
    raise ValueError(f"Dataset not found: {source_data_dir}")

# ------------------------------------------------------------------
# Find SPH files
# ------------------------------------------------------------------
sph_list = glob.glob(
    os.path.join(source_data_dir, "**", "*.sph"),
    recursive=True
)

print(f"Found {len(sph_list)} SPH files")

# ------------------------------------------------------------------
# WAV output directory
# ------------------------------------------------------------------
target_wavs_dir = os.path.join(target_data_dir, "wavs")
os.makedirs(target_wavs_dir, exist_ok=True)

# ------------------------------------------------------------------
# Skip conversion if WAV files already exist
# ------------------------------------------------------------------
existing_wavs = glob.glob(
    os.path.join(target_wavs_dir, "*.wav")
)

if len(existing_wavs) > 0:
    print(f"✅ Found {len(existing_wavs)} existing WAV files.")
    print("✅ Skipping audio conversion.")
else:
    print("Converting SPH files to WAV...")

    for sph_path in sph_list:

        wav_path = os.path.join(
            target_wavs_dir,
            os.path.splitext(
                os.path.basename(sph_path)
            )[0] + ".wav"
        )

        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-i",
                sph_path,
                wav_path,
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True,
        )

    print("✅ Audio conversion completed.")

# ------------------------------------------------------------------
# Build training manifest
# ------------------------------------------------------------------
train_transcripts = os.path.join(
    source_data_dir,
    "etc",
    "an4_train.transcription"
)

train_manifest = os.path.join(
    target_data_dir,
    "train_manifest.json"
)

an4_build_manifest(
    train_transcripts,
    train_manifest,
    target_wavs_dir
)

# ------------------------------------------------------------------
# Build test manifest
# ------------------------------------------------------------------
test_transcripts = os.path.join(
    source_data_dir,
    "etc",
    "an4_test.transcription"
)

test_manifest = os.path.join(
    target_data_dir,
    "test_manifest.json"
)

an4_build_manifest(
    test_transcripts,
    test_manifest,
    target_wavs_dir
)

print("\n✅ Done!")
print("Train manifest:", train_manifest)
print("Test manifest :", test_manifest)

Found 1078 SPH files
✅ Found 1078 existing WAV files.
✅ Skipping audio conversion.


c:\Users\osaoudi\.conda\envs\nemo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✅ Done!
Train manifest: c:\Users\osaoudi\Desktop\EASPORTS\VoiceComAnalysis\FineTune/an4_converted\train_manifest.json
Test manifest : c:\Users\osaoudi\Desktop\EASPORTS\VoiceComAnalysis\FineTune/an4_converted\test_manifest.json


In [6]:
# change path of the file here
import os
import IPython.display as ipd
path = os.environ["DATA_DIR"] + '/an4_converted/wavs/an268-mbmg-b.wav'
ipd.Audio(path)

In [7]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
1
NVIDIA RTX 2000 Ada Generation Laptop GPU


In [8]:
import torch

device = torch.device("cuda:0")
print(torch.cuda.get_device_name(0))

NVIDIA RTX 2000 Ada Generation Laptop GPU


In [1]:
import nemo, torch, pytorch_lightning as pl
print("nemo.__version__  :", nemo.__version__)
print("nemo.__file__     :", nemo.__file__)
print("pl.__version__    :", pl.__version__)
print("torch.__version__ :", torch.__version__)
print("cwd               :", __import__('os').getcwd())

c:\Users\osaoudi\Desktop\EASPORTS\VoiceComAnalysis\FineTune\nemo-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


nemo.__version__  : 3.0.0
nemo.__file__     : c:\Users\osaoudi\Desktop\EASPORTS\VoiceComAnalysis\FineTune\nemo-venv\Lib\site-packages\nemo\__init__.py
pl.__version__    : 2.6.6
torch.__version__ : 2.6.0+cu124
cwd               : c:\Users\osaoudi\Desktop\EASPORTS\VoiceComAnalysis\FineTune
